<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/28_rag_confidence_calibration/rag_confidence_calibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers pandas numpy scikit-learn

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

In [21]:
documents = [
    "Elon Musk founded SpaceX.",
    "SpaceX works on rockets and space exploration.",
    "Tesla builds electric cars.",
    "Elon Musk is CEO of Tesla."
]

df = pd.DataFrame({"text": documents})

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(df["text"].tolist())

In [ ]:
def retrieve_with_scores(query, k=2):
    query_embedding = model.encode(query)
    scores = util.cos_sim(query_embedding, doc_embeddings)[0]

    top_idx = np.argsort(-scores)[:k]

    return df.iloc[top_idx], scores[top_idx]

In [ ]:
def calculate_confidence(scores):
    top_score = float(scores[0])

    if top_score > 0.7:
        return "High", top_score
    elif top_score > 0.5:
        return "Medium", top_score
    else:
        return "Low", top_score

In [ ]:
answers = ["Elon Musk", "SpaceX", "Tesla"]
answer_embeddings = model.encode(answers)

def extract_answer(query):
    query_embedding = model.encode(query)
    scores = util.cos_sim(query_embedding, doc_embeddings)[0].cpu().numpy()

    return answers[scores.argmax()]

In [ ]:
def confidence_rag(query):
    print("🔹 Query:", query)

    docs, scores = retrieve_with_scores(query)

    print("\n📄 Retrieved Docs:\n", docs)

    confidence_level, score = calculate_confidence(scores)

    answer = extract_answer(query)

    print("\n✅ Answer:", answer)
    print("📊 Confidence Level:", confidence_level)
    print("🔢 Score:", score)

    return answer, confidence_level

In [22]:
confidence_rag("Which company works on rockets?")

🔹 Query: Which company works on rockets?

📄 Retrieved Docs:
                                              text
1  SpaceX works on rockets and space exploration.
0                       Elon Musk founded SpaceX.

✅ Answer: SpaceX
📊 Confidence Level: Medium
🔢 Score: 0.6724112629890442


('SpaceX', 'Medium')